# Fine-Tune Open Weight Language Models for Function Calling in Strands Agents

[![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)

## Introduction

This notebook demonstrates how to fine-tune small language models (1-3B parameters) for reliable function calling in edge environments. We leverage optimized training pipelines to achieve 2-5x faster training with 50% less memory usage compared to standard implementations.

Function calling, also known as tool use, enables language models to interact with external systems through structured API calls. This is critical for edge deployment where models must control physical devices, query databases, or invoke services with high reliability.

### The Problem We're Solving

Edge devices in vehicles and industrial settings need AI assistants that can:
- **Understand natural language**: "It's too hot" → climate control
- **Execute actions locally**: No cloud dependency for critical controls
- **Work with limited resources**: 2-4GB RAM, no GPU required
- **Maintain high accuracy**: Safety-critical operations demand reliability

### Our Solution: Fine-Tuned Tool Calling

Instead of using a general-purpose model, we specialize Qwen3-1.7B for specific tools:

```mermaid
graph LR
    A[User Input:<br/>'Set the temperature to 72 degrees'] --> B[Model Recognition:<br/>Intent = climate_control<br/>Parameter = 72]
    B --> C[Strands Format:<br/>toolUse.name: climate_control<br/>toolUse.input.command: 'set to 72']
    C --> D[Agent Execution:<br/>Virtual ECU updates<br/>climate state]
    D --> E[User Feedback:<br/>'Temperature set to 72°F']
    
    style A fill:#e1f5fe
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
    style E fill:#fce4ec
```

### What You'll Learn

- Generate high-quality synthetic training data using teacher-student approaches
- Fine-tune models efficiently using LoRA (Low-Rank Adaptation)
- Quantize models for edge deployment while preserving accuracy
- Benchmark performance improvements systematically
- Deploy models using llama.cpp for production inference

### Understanding Model Quantization

Quantization is the process of reducing numerical precision to compress models while preserving performance. In neural networks, weights and activations typically use 32-bit (FP32) or 16-bit (FP16/BF16) floating-point numbers. Quantization reduces these to 8-bit integers or even 4-bit representations, achieving:

- **Size Reduction**: 4-bit quantization reduces model size by ~75% compared to FP16
- **Speed Improvement**: Integer operations are faster than floating-point on most hardware
- **Memory Efficiency**: Enables deployment on resource-constrained edge devices
- **Minimal Accuracy Loss**: Modern quantization methods preserve >99% of model quality

#### Quantization Methods

**K-means Quantization (K-quants)**: Groups similar weights into clusters, storing cluster indices instead of full values. The 'K' variants (Q4_K_M, Q5_K_M) use different bit allocations for different tensor components.

**Dynamic vs Static**: Dynamic quantization determines scale factors at runtime, while static uses pre-computed values. We use static for predictable edge performance.

**Mixed Precision**: Critical layers (like embeddings) maintain higher precision while less sensitive layers use aggressive quantization.

In [ ]:
%%capture
# Install dependencies
!pip install -q --upgrade pip
!pip install -q -e ../../[function-calling]

In [ ]:
# Imports
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, TaskType, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Data Preparation

### Optional: Generate Custom Training Data

This notebook includes pre-generated training data (651 examples) and test data (266 examples) that follows the correct format for function calling. However, if you want to generate additional training data or customize the examples, you can use the built-in data generator.

**Requirements for data generation:**
- AWS Bedrock access with Claude 3.5 Sonnet (for high-quality synthetic data)
- Proper AWS credentials configured
- Bearer token (if required by your setup)

**Skip this cell if you want to use the existing training data.**

In [ ]:
# OPTIONAL: Generate additional training data using AWS Bedrock
# Uncomment the lines below if you want to generate custom training data

# from utils.data_generator import DataGenerator, ToolRegistry

# Set your AWS Bearer Token
# os.environ['AWS_BEARER_TOKEN_BEDROCK'] = 'YOUR_ACTUAL_BEARER_TOKEN_HERE'

# Generate training data
# from utils.data_generator import DataGenerator
# generator = DataGenerator()
# generator.generate_dataset(
#     num_examples=500,
#     output_path="data/train.jsonl", 
#     format_for_training=True
# )

# Generate test data  
# generator.generate_dataset(
#     num_examples=100,
#     output_path="data/test.jsonl",
#     format_for_training=True
# )

In [ ]:
# Load training data
train_path = Path('data/train.jsonl')
test_path = Path('data/test.jsonl')

train_texts = []
with open(train_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        train_texts.append(data['text'])

test_texts = []
with open(test_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        test_texts.append(data['text'])

print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")

In [ ]:
# Exploratory Data Analysis of Training and Test Datasets
import matplotlib.pyplot as plt
import re
from collections import Counter

# Extract tool usage and text lengths from datasets
def analyze_dataset(texts, dataset_name):
    tool_counts = Counter()
    text_lengths = []
    bilingual_count = 0
    
    for text in texts:
        # Count tool usage
        tool_match = re.search(r'"name":\s*"([^"]+)"', text)
        if tool_match:
            tool_counts[tool_match.group(1)] += 1
        
        # Text length analysis
        text_lengths.append(len(text))
        
        # Check for Japanese/bilingual content
        if any(ord(char) > 127 for char in text):
            bilingual_count += 1
    
    return tool_counts, text_lengths, bilingual_count

# Analyze both datasets
train_tools, train_lengths, train_bilingual = analyze_dataset(train_texts, "Training")
test_tools, test_lengths, test_bilingual = analyze_dataset(test_texts, "Test")

# Create 4 EDA charts
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# Chart 1: Tool Distribution Comparison
tools = list(set(train_tools.keys()) | set(test_tools.keys()))
train_counts = [train_tools.get(tool, 0) for tool in tools]
test_counts = [test_tools.get(tool, 0) for tool in tools]

x = range(len(tools))
width = 0.35
ax1.bar([i - width/2 for i in x], train_counts, width, label='Training', alpha=0.8)
ax1.bar([i + width/2 for i in x], test_counts, width, label='Test', alpha=0.8)
ax1.set_xlabel('Tool Names')
ax1.set_ylabel('Count')
ax1.set_title('Tool Usage Distribution')
ax1.set_xticks(x)
ax1.set_xticklabels([tool.replace('_', '\n') for tool in tools], rotation=45)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Chart 2: Text Length Distribution
ax2.hist(train_lengths, bins=30, alpha=0.7, label=f'Training (n={len(train_texts)})', density=True)
ax2.hist(test_lengths, bins=30, alpha=0.7, label=f'Test (n={len(test_texts)})', density=True)
ax2.set_xlabel('Text Length (characters)')
ax2.set_ylabel('Density')
ax2.set_title('Text Length Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Chart 3: Dataset Size Comparison
datasets = ['Training', 'Test']
sizes = [len(train_texts), len(test_texts)]
colors = ['#1f77b4', '#ff7f0e']
bars = ax3.bar(datasets, sizes, color=colors, alpha=0.8)
ax3.set_ylabel('Number of Examples')
ax3.set_title('Dataset Sizes')
ax3.grid(True, alpha=0.3)

# Add value labels on bars
for bar, size in zip(bars, sizes):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 5,
             f'{size:,}', ha='center', va='bottom', fontweight='bold')

# Chart 4: Bilingual Content Analysis
bilingual_data = [
    ['English Only', 'Bilingual (EN+JP)'],
    [len(train_texts) - train_bilingual, train_bilingual],
    [len(test_texts) - test_bilingual, test_bilingual]
]

x = range(len(bilingual_data[0]))
width = 0.35
ax4.bar([i - width/2 for i in x], bilingual_data[1], width, label='Training', alpha=0.8)
ax4.bar([i + width/2 for i in x], bilingual_data[2], width, label='Test', alpha=0.8)
ax4.set_xlabel('Content Type')
ax4.set_ylabel('Count')
ax4.set_title('Language Distribution')
ax4.set_xticks(x)
ax4.set_xticklabels(bilingual_data[0])
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"Training: {len(train_texts):,} examples, {train_bilingual} bilingual ({train_bilingual/len(train_texts)*100:.1f}%)")
print(f"Test: {len(test_texts):,} examples, {test_bilingual} bilingual ({test_bilingual/len(test_texts)*100:.1f}%)")
print(f"Avg text length - Training: {sum(train_lengths)/len(train_lengths):.0f} chars, Test: {sum(test_lengths)/len(test_lengths):.0f} chars")

## Model Setup

In [ ]:
# Load base model
model_name = "Qwen/Qwen3-1.7B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")

In [ ]:
# Configure LoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model.enable_input_require_grads()

## Training

In [ ]:
# Prepare datasets
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=1024
    )

train_dataset = Dataset.from_dict({"text": train_texts})
test_dataset = Dataset.from_dict({"text": test_texts})

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir="./qwen3-function-calling",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

In [ ]:
# Train the model
trainer.train()
trainer.save_model("./qwen3-function-calling-final")
tokenizer.save_pretrained("./qwen3-function-calling-final")

## Training Results

In [ ]:
# Display training metrics
import matplotlib.pyplot as plt

# Extract loss from training history
train_loss = [log['loss'] for log in trainer.state.log_history if 'loss' in log]
eval_loss = [log['eval_loss'] for log in trainer.state.log_history if 'eval_loss' in log]

# Create loss plot
if train_loss or eval_loss:
    plt.figure(figsize=(10, 5))
    
    if train_loss:
        plt.subplot(1, 2, 1)
        plt.plot(train_loss)
        plt.title('Training Loss')
        plt.xlabel('Steps')
        plt.ylabel('Loss')
        plt.grid(True, alpha=0.3)
    
    if eval_loss:
        plt.subplot(1, 2, 2)
        plt.plot(eval_loss, 'orange')
        plt.title('Evaluation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Test & Evaluation

In [ ]:
# Compare base model vs fine-tuned model
from peft import PeftModel

def compare_models():
    """
    Compare base Qwen3 model vs fine-tuned model on tool calling tasks.
    Shows the improvement from fine-tuning.
    """
    
    # Load base model for comparison
    print("Loading base model for comparison...")
    base_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-1.7B",
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    base_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B", trust_remote_code=True)
    if base_tokenizer.pad_token is None:
        base_tokenizer.pad_token = base_tokenizer.eos_token
    
    # Load fine-tuned model from saved checkpoint
    print("Loading fine-tuned model from checkpoint...")
    ft_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-1.7B",
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    ft_model = PeftModel.from_pretrained(ft_model, "./qwen3-function-calling-final")
    ft_tokenizer = AutoTokenizer.from_pretrained("./qwen3-function-calling-final", trust_remote_code=True)
    if ft_tokenizer.pad_token is None:
        ft_tokenizer.pad_token = ft_tokenizer.eos_token
    
    # Test prompt with proper format matching training data
    test_prompt = """<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "climate_control", "description": "Control vehicle climate settings", "inputSchema": {"json": {"type": "object", "properties": {"command": {"type": "string", "description": "Climate Control command"}}, "required": ["command"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Set the temperature to 72 degrees
<|im_end|>
<|im_start|>assistant
"""
    
    # Test base model
    print("\n" + "="*50)
    print("BASE MODEL OUTPUT:")
    print("="*50)
    
    inputs = base_tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(base_model.device) for k, v in inputs.items()}
    
    base_model.eval()
    with torch.no_grad():
        base_outputs = base_model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.7,
            do_sample=True,
            pad_token_id=base_tokenizer.pad_token_id
        )
    
    base_response = base_tokenizer.decode(base_outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    print(base_response[:500] if len(base_response) > 500 else base_response)
    
    # Check if base model produces correct format
    base_has_tool = "<tool_call>" in base_response
    base_has_json = "climate_control" in base_response
    
    # Test fine-tuned model
    print("\n" + "="*50)
    print("FINE-TUNED MODEL OUTPUT:")
    print("="*50)
    
    inputs = ft_tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(ft_model.device) for k, v in inputs.items()}
    
    ft_model.eval()
    with torch.no_grad():
        ft_outputs = ft_model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.7,
            do_sample=True,
            pad_token_id=ft_tokenizer.pad_token_id
        )
    
    ft_response = ft_tokenizer.decode(ft_outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    print(ft_response[:500] if len(ft_response) > 500 else ft_response)
    
    # Check if fine-tuned model produces correct format
    ft_has_tool = "<tool_call>" in ft_response
    ft_has_json = "climate_control" in ft_response
    
    # Summary
    print("\n" + "="*50)
    print("COMPARISON SUMMARY:")
    print("="*50)
    print(f"Base Model:")
    print(f"  - Has tool call tags: {'✓' if base_has_tool else '✗'}")
    print(f"  - Has correct tool:   {'✓' if base_has_json else '✗'}")
    print(f"\nFine-tuned Model:")
    print(f"  - Has tool call tags: {'✓' if ft_has_tool else '✗'}")
    print(f"  - Has correct tool:   {'✓' if ft_has_json else '✗'}")
    
    # Clean up model memory
    del base_model
    del ft_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    return base_response, ft_response

# Run comparison
base_output, finetuned_output = compare_models()

## Export Model

In [ ]:
# Merge LoRA weights and save
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./qwen3-function-calling-merged")

## Evaluation with Strands Agents

Start llama.cpp server first:
```bash
./llama-server -m qwen3-finetuned.gguf --host 0.0.0.0 --port 8080 -c 2048 --jinja
```

In [ ]:
# [OPTIONAL] Import production tools for integration testing
# Import production tools for evaluation
import sys
import requests
sys.path.append('../..')  # Add project root

from strands import Agent
from strands.models.llamacpp import LlamaCppModel
from src.agents.cockpit import (
    climate_control,
    window_control,
    seat_control,
    lighting_control,
    drive_mode
)

PRODUCTION_TOOLS = [
    climate_control,
    window_control,
    seat_control,
    lighting_control,
    drive_mode
]

# Check if server is running
def check_server(port):
    try:
        response = requests.get(f"http://localhost:{port}/health", timeout=1)
        return response.status_code == 200
    except:
        return False

if check_server(8080):
    print("Server running on port 8080")
else:
    print("Start llama.cpp server on port 8080 first")

In [ ]:
# Evaluate with Strands Agent
def evaluate_with_strands(port=8080):
    if not check_server(port):
        print(f"Server not running on port {port}")
        return
    
    # Create model and agent
    model = LlamaCppModel(
        base_url=f"http://localhost:{port}",
        params={"temperature": 0.7, "max_tokens": 200}
    )
    
    agent = Agent(
        model=model,
        tools=PRODUCTION_TOOLS,
        system_prompt="You are a vehicle assistant with access to control tools."
    )
    
    # Test cases
    test_cases = [
        ("Set temperature to 72", "climate_control"),
        ("Open driver window", "window_control"),
        ("Switch to sport mode", "drive_mode"),
        ("Turn on headlights", "lighting_control"),
        ("Adjust seat forward", "seat_control")
    ]
    
    correct = 0
    for prompt, expected in test_cases:
        try:
            response = agent.run(prompt)
            response_str = str(response)
            
            # Check if correct tool was called
            tool_called = None
            for tool in PRODUCTION_TOOLS:
                if tool.__name__ in response_str:
                    tool_called = tool.__name__
                    break
            
            is_correct = tool_called == expected
            if is_correct:
                correct += 1
            
            status = "PASS" if is_correct else "FAIL"
            print(f"{status}: {prompt:30} -> {expected}")
            
        except Exception as e:
            print(f"ERROR: {prompt:30} -> {str(e)[:50]}")
    
    accuracy = (correct / len(test_cases)) * 100
    print(f"\nAccuracy: {accuracy:.1f}% ({correct}/{len(test_cases)})")

# Run evaluation if server is available
if check_server(8080):
    evaluate_with_strands(8080)